In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_curve, RocCurveDisplay, auc, roc_auc_score

In [2]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from rdkit import DataStructs
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import MACCSkeys

In [3]:
import matplotlib.pyplot as plt

In [12]:
df1 = pd.read_csv('idotdo_final_871.csv')
df1.dropna(ignore_index=True, inplace=True)
df1

,CHEMBLID,smiles,ido_ic50,tdo_ic50
0,CHEMBL1098875,O=C1c2cc(F)ccc2-n2c1nc1ccccc1c2=O,7.638272,7.221849
1,CHEMBL1209728,Cc1c(Br)oc2c1C(=O)C(=O)c1c-2ccc2c1CCCC2(C)C,5.000000,5.000000
2,CHEMBL1276265,O=C1c2ccc(Cl)cc2-n2c1nc1ccccc1c2=O,6.737549,3.649364
3,CHEMBL1346056,Oc1ccccc1-c1nc2c3ccccc3c3ccccc3c2[nH]1,4.928486,4.911864
4,CHEMBL139935,O=[N+]([O-])c1cc(F)c2cccnc2c1O,4.835350,4.519562
...,...,...,...,...
866,43a,CS(=O)(NC1=CC=C(C2=CC3=C(C=N2)C(O)=C(OC)C=C3)C...,5.906578,6.698970
867,43b,O=S(C(F)(F)F)(NC1=CC=C(C2=CC3=C(C=N2)C(O)=C(OC...,6.508638,7.096910
868,43c,O=S(C1CC1)(NC2=CC=C(C3=CC4=C(C=N3)C(O)=C(OC)C=...,5.342944,5.473661
869,43d,O=S(N1CCCCC1)(NC2=CC=C(C3=CC4=C(C=N3)C(O)=C(OC...,4.813326,5.098542


In [15]:
df3 = df1.loc[:, ['ido_ic50', 'tdo_ic50']]
df3

,ido_ic50,tdo_ic50
0,7.638272,7.221849
1,5.000000,5.000000
2,6.737549,3.649364
3,4.928486,4.911864
4,4.835350,4.519562
...,...,...
866,5.906578,6.698970
867,6.508638,7.096910
868,5.342944,5.473661
869,4.813326,5.098542


In [16]:
for i in range(760):
    if df3['ido_ic50'].values[i] >= 6.15:
        df3.loc[i, ['ido_ic50']] = 1
    else:
        df3.loc[i, ['ido_ic50']] = 0

In [17]:
for i in range(760):
    if df3['tdo_ic50'].values[i] >= 6.0:
        df3.loc[i, ['tdo_ic50']] = 1
    else:
        df3.loc[i, ['tdo_ic50']] = 0

In [18]:
df3

,ido_ic50,tdo_ic50
0,1.000000,1.000000
1,0.000000,0.000000
2,1.000000,0.000000
3,0.000000,0.000000
4,0.000000,0.000000
...,...,...
866,5.906578,6.698970
867,6.508638,7.096910
868,5.342944,5.473661
869,4.813326,5.098542


In [21]:
newcol = []
for i in range(len(df3)):
    if df3['ido_ic50'].values[i] == 1 and df3['tdo_ic50'].values[i] == 1:
        newcol.append('AA')
    elif df3['ido_ic50'].values[i] == 0 and df3['tdo_ic50'].values[i] == 0:
        newcol.append('II')
    elif df3['ido_ic50'].values[i] == 0 and df3['tdo_ic50'].values[i] == 1:
        newcol.append('IA')
    else:
        newcol.append('AI')

In [22]:
target = pd.DataFrame(data=newcol, columns=['ido_tdo'])
target

,ido_tdo
0,AA
1,II
2,AI
3,II
4,II
...,...
866,AI
867,AI
868,AI
869,AI


In [23]:
y = target.values.reshape(len(df3),)
y.shape

(871,)

In [24]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [25]:
y_label = encoder.fit_transform(y)

In [26]:
y_label

array([0, 3, 1, 3, 3, 3, 2, 3, 3, 3, 2, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1,
       3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 0, 2, 3, 3, 1, 2, 1, 2, 3, 3, 2, 3,
       3, 2, 2, 2, 2, 0, 2, 0, 2, 2, 2, 2, 2, 2, 2, 3, 1, 3, 3, 3, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 0, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1,
       3, 1, 1, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 2, 0, 1, 3, 0, 0,
       1, 3, 0, 2, 1, 3, 3, 1, 0, 1, 0, 0, 0, 0, 1, 3, 1, 0, 0, 0, 1, 0,
       3, 1, 0, 0, 1, 0, 2, 0, 0, 0, 1, 0, 0, 1, 1, 3, 0, 0, 3, 0, 0, 3,
       0, 1, 2, 1, 0, 0, 0, 0, 2, 2, 1, 0, 1, 0, 3, 0, 0, 0, 0, 0, 3, 0,
       0, 1, 1, 2, 0, 0, 0, 1, 0, 0, 3, 1, 1, 2, 0, 3, 3, 0, 0, 3, 0, 3,
       0, 3, 3, 3, 0, 0, 0, 1, 1, 1, 1, 0, 2, 0, 3, 0, 0, 0, 1, 1, 1, 0,
       1, 0, 0, 0, 3, 0, 3, 3, 3, 2, 2, 0, 1, 0, 2, 3, 0, 0, 1, 0, 0, 0,
       3, 3, 0, 0, 3, 3, 1, 2, 3, 0, 3, 0, 2, 0, 1, 3, 1, 0, 0, 3, 3, 0,
       0, 2, 1, 0, 3, 0, 3, 3, 0, 3, 3, 1, 1, 3, 3,

In [27]:
plt.hist(y_label)

(array([208.,   0.,   0., 282.,   0.,   0.,  80.,   0.,   0., 301.]),
 array([0. , 0.3, 0.6, 0.9, 1.2, 1.5, 1.8, 2.1, 2.4, 2.7, 3. ]),
 <BarContainer object of 10 artists>)

In [28]:
from rdkit.Chem import SaltRemover as sr
remover = sr.SaltRemover()

In [29]:
mols = []
for i in range(len(df1)):
    try:
        mol_i = Chem.MolFromSmiles(df1['smiles'][i])
        if mol_i is None:
            print(f"[WARNING] Invalid SMILES at index {i}, skipping.")
            mols.append(None)  # Maintain indexing for multiprocessing
            continue
        mol_i = remover.StripMol(mol_i, dontRemoveEverything=True)
        mols.append(mol_i)
    except Exception as e:
        print(e)
len(mols)

871

In [30]:
smiles_list = [df1['smiles'][i] for i in range(len(df1['smiles']))]

In [33]:
#Morgan Fingerprints:
fpgen2 = AllChem.GetMorganGenerator(fpSize=1024)
i = 0
l2 = np.zeros((1, 1024), dtype='uint8')
for mol in mols:
    fp2 = fpgen2.GetFingerprintAsNumPy(mols[i])
    l2 = np.vstack((l2, fp2))
    i += 1
MFPs = l2[1:, :]
MFPs.shape

(871, 1024)

In [34]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=256, random_state=42)
X = svd.fit_transform(MFPs)

In [43]:
x_train, x_test, y_train, y_test = train_test_split(X, y_label, test_size=0.2, random_state=42)
x_test.shape

(175, 256)

In [44]:
x_train.shape

(696, 256)

In [45]:
x_test.shape

(175, 256)

In [41]:
from sklearn.model_selection import GridSearchCV

In [46]:
model = RandomForestClassifier()
params = {'criterion': ['gini', 'entropy'],  'max_depth': list(range(3, 20)), 'max_features': ['sqrt', 'log2', None], 'bootstrap': [True, False]}
grid = GridSearchCV(model, params, cv=10, n_jobs=-1)
grid.fit(x_train, y_train)

GridSearchCV(cv=10, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'bootstrap': [True, False],
                         'criterion': ['gini', 'entropy'],
                         'max_depth': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,
                                       15, 16, 17, 18, 19],
                         'max_features': ['sqrt', 'log2', None]})

In [47]:
#With 256 dimension
grid.best_estimator_

RandomForestClassifier(bootstrap=False, criterion='entropy', max_depth=14)

In [48]:
#With 256 dimension
grid.best_params_

{'bootstrap': False,
 'criterion': 'entropy',
 'max_depth': 14,
 'max_features': 'sqrt'}

In [49]:
#With 256 dimension
model = RandomForestClassifier(max_depth=14, bootstrap=False, criterion='entropy', random_state=42)

In [88]:
#With 256 dimension
np.mean(cross_val_score(model, x_train, y_train, cv=10))

0.6980684811237927

In [50]:
#With 256 dimension
model.fit(x_train, y_train)

RandomForestClassifier(bootstrap=False, criterion='entropy', max_depth=14,
                       random_state=42)

In [51]:
#With 256 dimension
model.score(x_train, y_train)

0.9942528735632183

In [52]:
#With 256
model.score(x_test, y_test)

0.7371428571428571

In [54]:
py_test = model.predict(x_test)

In [55]:
#With 256 dimension
print(classification_report(y_test, py_test))

              precision    recall  f1-score   support

           0       0.62      0.67      0.64        36
           1       0.86      0.71      0.78        59
           2       0.73      0.47      0.57        17
           3       0.72      0.87      0.79        63

    accuracy                           0.74       175
   macro avg       0.73      0.68      0.70       175
weighted avg       0.75      0.74      0.73       175



In [56]:
#With 256 dimension
roc_auc_score(y_test, model.predict_proba(x_test), multi_class='ovr', average='macro')

0.909684606225837

In [57]:
#With 256 dimension
roc_auc_score(y_test, model.predict_proba(x_test), multi_class='ovr', average=None)

array([0.89808153, 0.9191993 , 0.91995532, 0.90150227])

In [58]:
x_train.shape, y_train.shape, x_test.shape

((696, 256), (696,), (175, 256))